# INSTRUCTOR SOLUTIONS / DO NOT DISTRIBUTE

# Lesson 08: Neural Networks Walkthrough - Solutions
## Medina County Career Center - Applications of Artificial Intelligence

**Objective:** Understand how neural networks work and build one in Python.

**By the end of this lesson, you will:**
- Explain what a neuron is and how weights work
- Understand forward pass and backpropagation (conceptually)
- Build a neural network using scikit-learn
- Compare neural networks to linear regression
- Identify when to use neural networks

## Sub-Lesson 08a — How Neural Networks Work

### Part 1: The Neuron Analogy

Your brain has ~86 billion neurons. Each one:
1. **Receives signals** from other neurons
2. **Adds them up** (some signals are stronger/"weighted" than others)
3. **Fires or doesn't fire** based on whether the total exceeds a threshold

An artificial neuron does the same thing with numbers!

```
Biological Neuron          Artificial Neuron
  (soma)                        (node)
    ↑ receives              receives inputs
    |                        ↓
    | signal = weight ×     multiply by weights
    |              input    ↓
    | (dendrites)           sum them up
    ↑                        ↓
(axon fires if sum           apply activation
exceeds threshold)          function (threshold)
    ↓                        ↓
  send signal              output to next layer
```

### Part 2: Weights and Bias

Each connection has a **weight** - a number that gets multiplied by the input.

```
Inputs        Weights        Sum with bias       Activation    Output
  x₁ ——(w₁)——┐
             ├—→[ Σ + b ]—→[Activate?]—→ output
  x₂ ——(w₂)——┘

Formula: output = activate( (x₁×w₁ + x₂×w₂ + ... ) + bias )
```

**During training:** The network learns the best weights to minimize error. Weights start random, then adjust thousands of times.

### Part 3: Layers and Architecture

Neurons organize into **layers**:

```
Input Layer       Hidden Layer        Output Layer
(raw data)        (learns patterns)   (prediction)

  temp_min ●                  ●
           |                 /|\         ●
  humidity ●———————————————●  | prediction
           |                \|/         
  wind ●                  ●
  
  (3 inputs)         (10 neurons)   (1 output)
```

**Input Layer:** Your raw data (temperature, humidity, wind speed)

**Hidden Layers:** Learn increasingly complex patterns. More hidden layers = deeper network = can learn more complex patterns

**Output Layer:** Final prediction

**Example configurations:**
- `(10,)` = 1 hidden layer with 10 neurons
- `(10, 5)` = 2 hidden layers: 10 neurons, then 5 neurons
- `(100, 50, 25)` = 3 hidden layers: 100, 50, 25 neurons (deep learning!)

### Part 4: How Training Works (Conceptual)

When a neural network trains, it repeats this cycle:

**1. Forward Pass**
```
Input Data → Apply weights → Activate → Prediction
```
The network makes a guess using current weights.

**2. Calculate Loss**
```
Loss = How wrong was the prediction?
Example: Predicted 75°F, actual was 73°F → Loss = 2°F
```

**3. Backpropagation**
```
Trace the error backwards through the network.
Figure out which weights caused the error.
Adjust them slightly.
```

**4. Repeat**
```
Do this thousands of times with different batches of training data.
```

It's like studying with practice tests: take a test, see what you got wrong, adjust your studying, take another test, repeat!

**Key idea:** We don't program the weights. The network **learns** them from data.

### Part 5: TensorFlow Playground (Hands-On)

**Visit:** [playground.tensorflow.org](https://playground.tensorflow.org)

This website lets you visually build and train neural networks without writing code!

#### Experiment 1: Simple Dataset
1. Make sure the **circle dataset** is selected (bottom left)
2. Set hidden layers to just **1 layer with 2 neurons**
3. Click **Play** and watch it train
4. **Question:** Did 2 neurons learn the circle pattern? Why or why not?
   **Answer:** 2 neurons are not enough. You'll see a single curved line, not a circle. Circles are more complex patterns.

#### Experiment 2: Add More Neurons
1. Stop the training
2. Increase to **1 layer with 8 neurons**
3. Click **Play** again
4. **Question:** Does it learn faster and better than 2 neurons?
   **Answer:** Yes! 8 neurons can represent more complex decision boundaries. It should solve the circle in fewer iterations.

#### Experiment 3: Add Hidden Layers
1. Click the **+** button to add another hidden layer
2. Set to **2 layers: 8 neurons each**
3. Click **Play**
4. **Question:** Is the network learning better? Faster? Why might extra layers help?
   **Answer:** Multiple layers can combine simpler patterns into more complex ones. This hierarchical approach often works better.

#### Experiment 4: The Hard Problem
1. Switch to the **spiral dataset** (the hardest one!)
2. Try **1 layer with 5 neurons** - does it learn?
   **Answer:** No, it struggles. The spiral is too complex for one shallow layer.
3. Increase to **2 layers with 16 neurons each** - better?
   **Answer:** Much better! The deeper network can capture the spiral's complex, intertwined pattern.
4. **Question:** Why does the spiral need a more complex network?
   **Answer:** The spiral's pattern is highly non-linear and intertwined. Simple linear or shallow networks can't represent it. More layers and neurons give the network capacity to learn this complexity.

**Key observations:**
- More neurons = more capacity to learn complex patterns
- More layers = can learn deeper abstractions
- But too many = slow training, overfitting, wasted computation
- Different problems need different architectures!

## Sub-Lesson 08b — Neural Networks in Python

### Part 1: Setup and Load Data

Let's build a neural network to predict weather!

In [ ]:
# Import all the libraries we need
import pandas as pd  # pandas = work with data tables (like Excel in Python)
import numpy as np  # numpy = math and arrays
from sklearn.model_selection import train_test_split  # split data into train/test
from sklearn.preprocessing import StandardScaler  # scale features to similar range
from sklearn.neural_network import MLPRegressor  # neural network for predictions
from sklearn.linear_model import LinearRegression  # for comparison
from sklearn.metrics import r2_score, mean_absolute_error  # evaluate performance
import warnings
warnings.filterwarnings('ignore')  # suppress warning messages

print("All libraries imported successfully!")

In [ ]:
# Load the Medina County weather data
# This is the SAME data from Lesson 07 (Linear Regression)
weatherData = pd.read_csv('medina_weather_2024.csv')

print(f"Data loaded! Shape: {weatherData.shape}")
print(f"\nFirst few rows:")
print(weatherData.head())
print(f"\nColumns: {list(weatherData.columns)}")

In [ ]:
# Prepare features (X) and target (y)
# X = inputs: temp_min, humidity, wind_speed
# y = what we're predicting: temp_max

X = weatherData[['temp_min', 'humidity', 'wind_speed']]  # features
y = weatherData['temp_max']  # target

print(f"Features X shape: {X.shape}")
print(f"Target y shape: {y.shape}")
print(f"\nFeature statistics:")
print(X.describe())

In [ ]:
# Split data into training and test sets
# Training set = teach the network (80% of data)
# Test set = see how it performs on new data (20% of data)
X_train, X_test, y_train, y_test = train_test_split(
    X,              # features
    y,              # target
    test_size=0.2,  # use 20% for testing
    random_state=42 # for reproducibility
)

print(f"Training set: {X_train.shape[0]} examples")
print(f"Test set: {X_test.shape[0]} examples")

### Part 2: Why Scale Data? (CRITICAL for Neural Networks!)

Neural networks learn by adjusting weights. If inputs have **very different scales**, training becomes slow and unstable.

**Example without scaling:**
- Temperature range: 0-100 (range of 100)
- Humidity range: 0-100 (range of 100)
- Wind speed range: 0-30 (range of 30)

The weights for wind_speed would need to be 3-4x larger than others to matter! This confuses the network.

**StandardScaler solution:**
- Transform all features so mean = 0, standard deviation = 1
- All features on similar scale: roughly -2 to +2
- Network learns much faster and better!

This is not optional - neural networks NEED scaled data!

In [ ]:
# Create a StandardScaler
# This learns the mean and std dev from training data
scaler = StandardScaler()

# Fit scaler on TRAINING data only, then transform
# (Never fit scaler on test data - that's data leakage!)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # use same scaling

print("Before scaling:")
print(f"  First training example: {X_train.iloc[0].values}")
print(f"  Mean: {X_train.mean().values}")
print(f"  Std Dev: {X_train.std().values}")

print("\nAfter scaling:")
print(f"  First training example: {X_train_scaled[0]}")
print(f"  Mean: {X_train_scaled.mean(axis=0)}")
print(f"  Std Dev: {X_train_scaled.std(axis=0)}")
print("\nNotice: Mean is ~0, Std Dev is ~1. Much better for neural networks!")

### Part 3: Build and Train the Neural Network

In [ ]:
# Create a neural network model
# MLPRegressor = Multi-Layer Perceptron Regressor (fancy name for neural network)

nnModel = MLPRegressor(
    hidden_layer_sizes=(10, 5),  # Architecture: 2 hidden layers with 10 and 5 neurons
    max_iter=1000,               # Train for up to 1000 iterations (epochs)
    random_state=42,             # for reproducibility
    verbose=0                     # don't print training details
)

print("Neural network created!")
print(f"Architecture: Input(3) → Hidden1(10) → Hidden2(5) → Output(1)")
print(f"\nTraining...")

# Train the network on scaled data
nnModel.fit(X_train_scaled, y_train)

print(f"Training complete! Trained for {nnModel.n_iter_} iterations")

### Part 4: Evaluate the Neural Network

In [ ]:
# Make predictions on test set (remember: use SCALED test data)
nnPredictions = nnModel.predict(X_test_scaled)

# Calculate performance metrics
nnR2 = r2_score(y_test, nnPredictions)
nnMAE = mean_absolute_error(y_test, nnPredictions)

print("Neural Network Performance:")
print(f"  R² Score: {nnR2:.4f}")
print(f"    (Interpretation: Explains {nnR2*100:.1f}% of variance in temperature)")
print(f"  Mean Absolute Error: {nnMAE:.2f}°F")
print(f"    (On average, predictions are off by {nnMAE:.1f}°F)")

### Part 5: Compare to Linear Regression (from Lesson 07)

Neural networks are powerful, but are they better than simpler models for this problem?

In [ ]:
# Build a Linear Regression model for comparison
# Note: Linear Regression does NOT need scaling
lrModel = LinearRegression()
lrModel.fit(X_train, y_train)  # use UNSCALED training data
lrPredictions = lrModel.predict(X_test)  # use UNSCALED test data

# Calculate Linear Regression metrics
lrR2 = r2_score(y_test, lrPredictions)
lrMAE = mean_absolute_error(y_test, lrPredictions)

print("Linear Regression Performance:")
print(f"  R² Score: {lrR2:.4f}")
print(f"  Mean Absolute Error: {lrMAE:.2f}°F")

In [ ]:
# Side-by-side comparison
print("\n" + "="*50)
print("COMPARISON: Linear Regression vs Neural Network")
print("="*50)
print(f"{'Model':<25} {'R² Score':<15} {'MAE (°F)':<10}")
print("-" * 50)
print(f"{'Linear Regression':<25} {lrR2:<15.4f} {lrMAE:<10.2f}")
print(f"{'Neural Network (10-5)':<25} {nnR2:<15.4f} {nnMAE:<10.2f}")
print("="*50)

if nnR2 > lrR2:
    improvement = ((nnR2 - lrR2) / lrR2) * 100
    print(f"\nNeural Network improved R² by {improvement:.1f}%")
elif lrR2 > nnR2:
    improvement = ((lrR2 - nnR2) / nnR2) * 100
    print(f"\nLinear Regression outperformed by {improvement:.1f}%")
else:
    print(f"\nBoth models perform equally!")

### Part 6: Interpretation and Insights

**Key Observations:**

1. **Similar Performance:** For this simple weather prediction problem, Linear Regression and Neural Networks perform very similarly. Why?
   - The relationship between temperature and weather features is mostly **linear**
   - We only have 3 input features
   - 365 training examples is decent but not huge

2. **When Neural Networks Shine:**
   - **Images:** CNN networks with thousands of features
   - **Text:** RNNs and Transformers with language patterns
   - **Large Datasets:** 1000s or millions of examples
   - **Complex Non-Linear Patterns:** When relationships are curved, not straight
   - **Multiple Hidden Layers:** Deep learning finds abstract patterns

3. **When Linear Regression is Better:**
   - Quick and simple
   - Interpretable: "For every 1°F increase in min temp, max temp increases by X°F"
   - Less data needed
   - Faster to train
   - No scaling required

**Bottom Line:** Choose the simplest model that solves your problem. Don't use a neural network if linear regression works!

---

### Part 7: Experiment with Architecture

#### Instructor Note:
Students should try changing hidden_layer_sizes to see effects:
- Smaller (5,): Underfitting - not enough capacity
- Larger (50, 25, 10): Overfitting risk on this small dataset, but may see small improvements
- The key lesson: More is not always better; simple models often generalize better

In [ ]:
# EXAMPLE: Try a simpler architecture
nnModel2 = MLPRegressor(hidden_layer_sizes=(5,), max_iter=1000, random_state=42)
nnModel2.fit(X_train_scaled, y_train)
nnPredictions2 = nnModel2.predict(X_test_scaled)
nnR2_2 = r2_score(y_test, nnPredictions2)
nnMAE_2 = mean_absolute_error(y_test, nnPredictions2)

print("\nARCHITECTURE COMPARISON:")
print(f"{'Architecture':<20} {'R² Score':<15} {'MAE (°F)':<10}")
print("-" * 45)
print(f"{'1 layer, 5 neurons':<20} {nnR2_2:<15.4f} {nnMAE_2:<10.2f}")
print(f"{'2 layers, 10-5':<20} {nnR2:<15.4f} {nnMAE:<10.2f}")

if nnR2 > nnR2_2:
    print(f"\nConclusion: Extra layer helped! R² improved by {((nnR2-nnR2_2)/nnR2_2)*100:.2f}%")
else:
    print(f"\nConclusion: Simpler model works better. Simpler = less overfitting risk!")

In [ ]:
# EXAMPLE: Try a bigger architecture
nnModel3 = MLPRegressor(hidden_layer_sizes=(50, 25, 10), max_iter=1000, random_state=42)
nnModel3.fit(X_train_scaled, y_train)
nnPredictions3 = nnModel3.predict(X_test_scaled)
nnR2_3 = r2_score(y_test, nnPredictions3)
nnMAE_3 = mean_absolute_error(y_test, nnPredictions3)

print("\nCOMPLETER ARCHITECTURE COMPARISON:")
print(f"{'Architecture':<25} {'R² Score':<15} {'MAE (°F)':<10}")
print("-" * 50)
print(f"{'1 layer, 5 neurons':<25} {nnR2_2:<15.4f} {nnMAE_2:<10.2f}")
print(f"{'2 layers, 10-5':<25} {nnR2:<15.4f} {nnMAE:<10.2f}")
print(f"{'3 layers, 50-25-10':<25} {nnR2_3:<15.4f} {nnMAE_3:<10.2f}")

print("\n### Instructor Note:")
print("Notice that bigger networks don't always perform better!")
print("With only 365 examples, complex architectures risk overfitting.")
print("The sweet spot is often a simpler model that generalizes well.")

## Summary

**What You Learned:**

✓ How artificial neurons work and why they need weights
✓ Neural network architecture (input, hidden, output layers)
✓ Forward pass and backpropagation (conceptually)
✓ Why scaling data is critical for neural networks
✓ How to build a neural network in scikit-learn
✓ How to compare neural networks to linear regression
✓ When to choose which model

**Key Takeaways:**
- Neural networks learn from data by adjusting millions of small weights
- Scaling features with StandardScaler is essential
- Simpler models (Linear Regression) often work better for simple problems
- Neural networks excel with images, audio, text, and complex non-linear patterns
- Always evaluate multiple models on the same problem!

**### Instructor Note:**
This walkthrough emphasizes the practical reality: not every problem needs a complex neural network. Linear regression outperformed or matched the neural network on this dataset, teaching students critical thinking about model selection.